In [ ]:
# General imports:-

import numpy as np;
import pandas as pd;
from warnings import filterwarnings;
filterwarnings('ignore');
from itertools import product;
from datetime import datetime;
from gc import collect;
from termcolor import colored;
from tqdm.notebook import tqdm;
from IPython.display import clear_output;

import matplotlib.pyplot as plt;
from seaborn import lineplot, heatmap;
%matplotlib inline

grid_specs = {'visible':True,'which':'both','color':'lightgrey','linestyle':'--','linewidth':0.50};
title_specs = {'color':'tab:blue', 'fontweight': 'bold', 'fontsize':12};

In [ ]:
# Model imports:-

from statsmodels.tsa.stattools import adfuller, kpss;
from dateutil.easter import easter;
from holidays import CountryHoliday;
from sklearn.pipeline import Pipeline, make_pipeline;
from sklearn.base import BaseEstimator, TransformerMixin;
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, RobustScaler;
from sklearn.model_selection import GroupKFold;
from sklearn.compose import ColumnTransformer;

## Tabular Playground Series- September 2022

* This competition aims to forecast the sales of 4 books across 6 European countries across 2 stores. 
* This is a time series regression problem with the SMAPE metric as the evaluation metric. 

**This is a model development notebook. We will explore the methods used in the high scoring public notebooks till date and also focus on traditional time series methods too**

* We use the below notebook as reference- https://www.kaggle.com/code/cabaxiom/tps-sep-22-eda-and-linear-regression-baseline/notebook#Modeling
* We also borrow ideas from my own EDA and visuals notebook. Link is as below- https://www.kaggle.com/code/ravi20076/tpssep22-eda-visualization

We firstly aggregate the sales across all stores and forecast the total sales for calendar year 2021. Once this is done, we split the total sales into the components. This idea is borrowed from the reference notebook.

In [ ]:
# Defining the competition metric and print functions:-
def CalcSMAPE(ytrue, ypred):
    """
    This function aims to calculate the Symmetric MAPE to be used as an evalutation metric in the competition. This is not directly available in scikit-learn.
    """;    
    
    SMAPE = abs(ytrue - ypred) / (abs(ytrue) + abs(ypred));
    SMAPE = SMAPE.mean() * 200;
    return SMAPE;

def PrintColor(text:str, color:str= 'blue', attrs:list = ['bold', 'dark']):
    "This function makes a colored print using the provided f-string";
    print(colored(text, color= color, attrs= attrs));

In [ ]:
# Initializing global list of important days for model development:-
imp_dates = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 16,17,22,
             124, 125, 126, 127, 140, 141, 167, 168, 
             169, 170, 171, 173, 174, 175, 176, 177, 178, 179, 
             180, 181, 203, 230, 231, 232, 233, 234, 282, 289, 
             290, 307, 308, 309, 310, 311, 312, 313, 317, 318, 
             319, 320, 360, 361, 362, 363, 364, 365];

# 1. Data processing

We import the reference data sets and prepare the data for the model in this section.

In [ ]:
xytrain = pd.read_csv('../input/tabular-playground-series-sep-2022/train.csv', index_col= 'row_id',
                      parse_dates=['date'],infer_datetime_format=True);
xtest = pd.read_csv('../input/tabular-playground-series-sep-2022/test.csv', index_col= 'row_id',
                      parse_dates=['date'],infer_datetime_format=True);

# Displaying the basic information and features:-
PrintColor(f"\nFeature list in the original training data");
Ftre_Lst = xtest.columns;
display(Ftre_Lst);

PrintColor(f"\nFeature information\n");
display(xytrain.info());

# Importing Sales Profile from the EDA output:-
Sales_Prf = pd.read_csv('../input/tpssep22-eda-visualization/Sales_Prf.csv', parse_dates = ['date'], index_col= 'date');
Comb_Ftre_Lst = Sales_Prf.columns[-48:];
PrintColor(f"\nFeature combination information from Sales Profile\n");
display(Comb_Ftre_Lst);

# Developing dates and monthids for effective plotting labels:-
Dt_Lbl = pd.DataFrame(Sales_Prf.index[Sales_Prf.index.is_quarter_end]);
Dt_Lbl = Dt_Lbl.assign(Monthid = Dt_Lbl.date.dt.year*100 + Dt_Lbl.date.dt.month);

In [ ]:
# Calculating and plotting the total sales across all products:-

ytrain = pd.DataFrame(np.sum(Sales_Prf[Comb_Ftre_Lst], axis=1), columns= ['num_sold']);
ytrain['LN_num_sold'] = np.log1p(ytrain);
ytrain['GR1day'] = ytrain['LN_num_sold'].diff(1);
ytrain['GR7day'] = ytrain['LN_num_sold'].diff(7);
ytrain['GR30day'] = ytrain['LN_num_sold'].diff(30);
ytrain['GR90day'] = ytrain['LN_num_sold'].diff(90);

In [ ]:
# The below function assesses trends and displays stationarity test results for the training columns:-
def AnalyzeGrowthSeries(ytrain:pd.DataFrame):
    "This function makes trend plots for the growth rates in the training data and evaluates stationarity test results too";
    
    fig, ax = plt.subplots(4,1,figsize= (30, 24));
    for i, col in enumerate(ytrain.columns[-4:]):
        a = ax[i];
        lineplot(y = ytrain[col], x= ytrain.index, color= 'tab:blue', ax=a);
        a.grid(**grid_specs);
        a.set_title(f"\nDynamics - {col}\n", **title_specs);
        a.set(xlabel= '', ylabel= '');
        del a;
    plt.tight_layout();
    plt.show();

    
    Stnty_Prf = pd.DataFrame(data= None, columns= ['Ftre','KPSS_C', 'KPSS_CT', 'ADF_C', 'ADF_CT', 'ADF_CTT'], dtype= np.float32);
    # Assessing stationarity for the growth rates:-
    for col in ytrain.columns[-5:]:
        results = [col];
        for method in ['c', 'ct']:
            results.extend([kpss(ytrain[col].dropna(), regression = method)[1]]) ;
        for method in ['c', 'ct', 'ctt']:
            results.extend([adfuller(ytrain[col].dropna(), regression = method)[1]]);

        results = pd.DataFrame(data= results).transpose();
        results.columns = Stnty_Prf.columns;
        Stnty_Prf = pd.concat([Stnty_Prf,results], axis=0, ignore_index= True);

    Stnty_Prf.set_index('Ftre', inplace= True);

    print(colored(f"\nStationarity information-- p-values\n", color= 'blue', attrs= ['bold', 'dark']));
    display(Stnty_Prf.style.format('{:.2%}'));

In [ ]:
filterwarnings('ignore');
AnalyzeGrowthSeries(ytrain);

In [ ]:
# Outlier removal:-
for col in ytrain.iloc[0:2, -4:].columns:
    Q1,Q3 = np.percentile(ytrain[col].dropna(),25),np.percentile(ytrain[col].dropna(),75);
    ytrain[col] = np.clip(ytrain[col], a_min = Q1 - 1.50*(Q3-Q1), a_max = Q3 + 1.50*(Q3-Q1));
    del Q3, Q1;

AnalyzeGrowthSeries(ytrain);

<div style="color:'#050a14';
           display:fill;
           border-radius:5px;
           background-color:#F5F5DC;
           font-size:110%;
           font-family:Calibri;
           letter-spacing:0.5px">

<p style="padding: 10px;color:'lightgrey'; font-weight: bold">
Key inferences:- 
    
1. Logarithm of sales is a non-stationary series
2. Growth rates are weakly stationary, except for certain tests. But this is fine overall, we may consider the series as stationary for our purposes
3. Cyclical trends are getting affected in H1 2020 due to covid, also, spikes are seen in December months too.
4. Post outlier correction, growth rate graphs look slightly better for modeling in my opinion
</p>
</div>

In [ ]:
# Calculating total daily sales by product:-
Ytrain = np.zeros(len(Sales_Prf))
for i in range(1,5,1):
    if i == 1: Ytrain += np.sum(Sales_Prf[Comb_Ftre_Lst[Comb_Ftre_Lst.str[2] == '1']], axis=1);
    else: Ytrain = np.c_[Ytrain,np.sum(Sales_Prf[Comb_Ftre_Lst[Comb_Ftre_Lst.str[2] == str(i)]], axis=1)];

Ytrain = pd.DataFrame(Ytrain, columns = range(1,5), index= Sales_Prf.index);

# Calculating product contribution rate:-
Ytrain['Total_Sales'] = np.sum(Ytrain, axis=1);
for i in range(1,5,1): Ytrain[f"ProductRatio{i}"] = Ytrain[i]/ Ytrain['Total_Sales'];
Ytrain.drop(['Total_Sales'], axis=1, inplace= True, errors= 'ignore');
      
# Plotting the daily sales volumes through the history:-
fig, ax = plt.subplots(2,1, figsize= (20, 16), sharex= True);

Ytrain.iloc[:,0:4].plot.line(color= ['tab:blue', 'orange', 'brown', 'black'], ax= ax[0]);
ax[0].set_title(f"\nDaily sales for 4 products\n", **title_specs);
ax[0].grid(**grid_specs);
ax[0].set(xlabel= '', ylabel= '');
ax[0].set_xticks(Dt_Lbl.date.values, labels= Dt_Lbl.Monthid, fontsize= 8, rotation= 45);

Ytrain.iloc[:,4:].plot.line(color= ['tab:blue', 'orange', 'brown', 'black'], ax= ax[1]);
ax[1].set_title(f"\nDaily sales contribution for 4 products\n", **title_specs);
ax[1].grid(**grid_specs);
ax[1].set(xlabel= '', ylabel= '');
ax[1].set_xticks(Dt_Lbl.date.values, labels= Dt_Lbl.Monthid, fontsize= 8, rotation= 45);

plt.tight_layout();
plt.show();
collect();

In [ ]:
# Calculating sales growth rate for forecast:-
_ = np.log1p(Ytrain.iloc[:, 0:4]);

# Calculating growth rates for sales:-
for i in [1,7,30,90]: Ytrain = pd.concat([Ytrain, _.diff(i).add_prefix(f'GR{i}day_')], axis=1);
    
Train_Ftre_Lst = Ytrain.columns[Ytrain.columns.str.startswith('GR').fillna(False)];
PrintColor(f"\nTraining features list for time series models\n");
display(Train_Ftre_Lst);

del _;
collect();

# 2. Feature Engineering

* In this section, we will develop a data pipeline from date features using the Sales Profile table used previously.
* We will use the aggregate Ytrain data set for the model, thus forecasting 4 time series only

## A. Stationarity Assessment:- 

We assess the stationarity of the aggregated time series in this sub-section using ADF and KPSS tests. 

In [ ]:
Stnty_Prf = pd.DataFrame(data= None, columns= ['Ftre','KPSS_C', 'KPSS_CT', 'ADF_C', 'ADF_CT', 'ADF_CTT'], dtype= np.float32);

# Assessing stationarity for the sales growth rates by product:-
for col in tqdm(Train_Ftre_Lst):
    results = [col];
    for method in ['c', 'ct']:
        results.extend([kpss(Ytrain[col].dropna(), regression = method)[1]]) ;
    for method in ['c', 'ct', 'ctt']:
        results.extend([adfuller(Ytrain[col].dropna(), regression = method)[1]]);

    results = pd.DataFrame(data= results).transpose();
    results.columns = Stnty_Prf.columns;
    Stnty_Prf = pd.concat([Stnty_Prf,results], axis=0, ignore_index= True);
    del results;
    collect();
collect();

Stnty_Prf["Is_Stationary"] =\
np.where((np.amax(Stnty_Prf[['KPSS_C', 'KPSS_CT']], axis=1) >= 0.10) | 
         (np.amin(Stnty_Prf.iloc[:, 2:], axis=1) <= 0.05),"Y","N");

PrintColor(f"\nAssessing stationarity of time series by ADF-KPSS tests\n");
display(Stnty_Prf.style.format(precision = 3));

In [ ]:
# Plotting specific series to assess trends (potential de-trending):-

fig, a = plt.subplots(2,1, sharex= True, figsize= (20,16));
for i, col in enumerate(['GR90day_1', 'GR90day_3']):
    ax = a[i];
    Ytrain[col].plot.line(color = 'tab:blue', ax= ax);
    ax.set_title(f"\n{col}\n", **title_specs);
    ax.grid(**grid_specs);
    ax.set(xlabel= '', ylabel= '');
    ax.set_xticks(ticks= Dt_Lbl.date.values, labels = Dt_Lbl.Monthid.values, rotation= 45);
    
plt.tight_layout();
plt.show();
collect();

In [ ]:
PrintColor(f"\nAssessing train-feature correlations\n");
display(Ytrain[Train_Ftre_Lst[Train_Ftre_Lst.str.startswith('GR1')]].dropna().corr().style.format('{:.2%}'));

<div style="color:'#050a14';
           display:fill;
           border-radius:5px;
           background-color:#F5F5DC;
           font-size:110%;
           font-family:Calibri;
           letter-spacing:0.5px">

<p style="padding: 10px;color:'lightgrey'; font-weight: bold">
Key inferences:- 
    
1. Ytrain data is developed with log-growth rates across 4-products for 1,7,30,90 days growth rates
2. These series appear stationarity with at least 1 test among KPSS and ADF
3. In some cases, de-trending may be needed, as seen with the KPSS and ADF p-values. Graphs also indicate trend relations for 90-day growths for products 1,3
4. The 4 time series are internally highly correlated, making a case for VARIMA models
</p>
</div>

Model development for this month's competition is largely influenced by the below approaches- 
1. https://www.kaggle.com/code/hosseinbehjat/no-need-to-aggregate-and-disaggregate-tpssep22/notebook
2. https://www.kaggle.com/code/cabaxiom/tps-sep-22-eda-and-linear-regression-baseline/notebook#Modeling

I tried to use traditional time series approaches to no good results, hence, switching to these methods in this release.

We use linear approaches like Ridge Regression, LASSO, elastic net and linear models for the same. 

## B. Data pipeline development-

In this sub-section, we develop a data pipeline using the Sales Profile data and the existing training data to elicit better features from the train-test data.

In [ ]:
class DateFtreCreator(BaseEstimator, TransformerMixin):
    "This class creates several features from the date column to be used subsequenntly for the model";  
    
    def __init__(self, imp_dates:list = imp_dates):
        self.imp_dates = imp_dates;

    def fit(self, X, y=None, **fit_params): 
        self.strt_year = min(X.date.dt.year) - 1;
        return self;
    
    def get_feature_names_in(self, X, y=None): return X.columns;
    
    def transform(self, X:pd.DataFrame, y=None, **transform_params):
        df = X.copy();
        
        df['Year_Nb'] = (df.date.dt.year - self.strt_year).astype(np.int8);
        df['Qtr_Nb'] = df['date'].dt.quarter.astype(np.int8);
        df["Month_Nb"] = df["date"].dt.month.astype(np.int8);
        df["Month_Sin"] = (np.sin(df['Month_Nb'] * (2 * np.pi / 12))).astype(np.float32);
        df["Day_Nb"] = df["date"].dt.day.astype(np.int16);
        df["Day_Sin"] = (np.sin(df['Day_Nb'] * (2 * np.pi / 12))).astype(np.float32);
        df["Weekday_Nb"] = df["date"].dt.dayofweek.astype(np.int8);
        df['Week_Nb'] = np.clip(df['date'].dt.week, a_min = 0, a_max= 52);
        df['Is_Weekend'] = np.where(df['Weekday_Nb'] >= 5,1,0);
        
        df["DayofYear_Nb"] = df["date"].dt.dayofyear;
        
        # Adjusting the day of the year number for leap year:-
        df["DayofYear_Nb"] = \
        df.apply(lambda x: x["DayofYear_Nb"]-1 
                 if (x["date"] > pd.Timestamp("2020-02-29") and x["date"] < pd.Timestamp("2021-01-01"))  
                 else x["DayofYear_Nb"], axis=1);
        
        df['Is_ImpDate'] = df['DayofYear_Nb'].apply(lambda x: x if x in self.imp_dates else 0);
  
        # Analyzing Easter date:- 
        easter_date = df.date.apply(lambda date: pd.Timestamp(easter(date.year)));        
        for day in list(range(-5, 5)) + list(range(40, 48)):
            df[f'Easter_{day}'] = (df.date - easter_date).dt.days.eq(day);
         
        # Envisaging one-hot encoding for days around Easter:-        
        for col in df.columns :
            if 'Easter' in col : df = pd.get_dummies(df, columns = [col], drop_first=True);

        self.feature_names_out = df.columns;
        
        return df;
    
    def get_feature_names_out(self, X, y=None): return self.feature_names_out;      

In [ ]:
class HolidayMapper(BaseEstimator, TransformerMixin):
    "This class maps the holidays to the training and test data sets by country";
    
    def __init__(self, strt_yr:int= 2017, end_yr:int= 2021): 
        self.start_yr = strt_yr;
        self.end_yr = end_yr;
    
    def get_feature_names_in(self, X, y=None): return X.columns;
    
    def fit(self, X, y=None, **fit_params): 
        self.country = X.country.unique();
        return self;
    
    def transform(self, X:pd.DataFrame, y=None, **transform_params):
        df = X.copy();
        
        period = range(self.start_yr, self.end_yr+1,1);
        for i in self.country:
            holidays = CountryHoliday(i, years= period);
            df['Holiday_Nm'] = df['date'].map(holidays).fillna('Not Holiday');
            df['Is_Holiday'] = np.where(df.Holiday_Nm =='Not Holiday',0,1);
            df['Holiday_Nm'] = df['Holiday_Nm'].apply(lambda x: x if x != 'Asunción de la Virgen (Trasladado)' else 'Not Holiday');
        
        df = df.drop(["Month_Nb","Day_Nb","DayofYear_Nb"], axis=1, errors= 'ignore');
        self.features = df.columns;
        return df;
    
    def get_feature_names_out(self, X, y=None): return self.features;     

In [ ]:
class MakeLabelEncoder(BaseEstimator, TransformerMixin):
    "This class encodes the country, store and products using label encoder";
    
    def __init__(self):pass;
    
    def get_feature_names_in(self, X, y=None): return X.columns;
    
    def fit(self, X, y=None, **fit_params): return self;
    
    def transform(self, X:pd.DataFrame, y=None, **transform_params):
        df = X.copy();
        df['product'] = LabelEncoder().fit_transform(df['product']);
        df['country'] = LabelEncoder().fit_transform(df['country']);
        df['store'] = LabelEncoder().fit_transform(df['store']);
        self.features = df.columns;
        return df;
    
    def get_feature_names_out(self, X, y=None): return self.features;

In [ ]:
class DateColMaker(BaseEstimator, TransformerMixin):
    "This class drops the duplicates in the train-test sets and returns only date transforms";
    
    def __init__(self): pass
  
    def get_feature_names_in(self, X, y=None): return X.columns;
    
    def fit(self, X, y=None, **fit_params): return self;
    
    def transform(self, X, y=None, **transform_params):
        df = X.copy();
        df = df.drop(['country', 'store', 'product'], axis=1, errors= 'ignore').drop_duplicates();
        df = df.set_index(['date']);
        self.features = df.columns;
        return df;
        
    def get_feature_names_out(self, X, y=None): return self.features;

In [ ]:
xtrain, ytrain = xytrain[Ftre_Lst], xytrain['num_sold'];

# Creating the data pipeline:-
processor = Pipeline(verbose= True,
                     steps= [('MakeDtFtre', DateFtreCreator()),('MapHolidays', HolidayMapper()),
                             ('MakeDateCol',DateColMaker())]);

OneHotCols = ['Weekday_Nb', 'Week_Nb', 'Qtr_Nb', 'Is_ImpDate', 'Holiday_Nm'];
OHencoder = Pipeline(verbose= True, 
                   steps= [('OHEncode', 
                            ColumnTransformer([('Encode', OneHotEncoder(sparse=False, drop='first'),
                                                OneHotCols)], remainder= 'drop'))]);

# Implementing the pipeline to get train-test arrays:-
print();
Xtrain = processor.fit_transform(xtrain, ytrain);
Xtest = processor.transform(xtest);

# Implementing the encoder pipeline to get train-test arrays:-
_ = OHencoder.fit_transform(Xtrain);
DateCols = OHencoder.get_feature_names_out();
DateCols = [i[i.find('__')+2:] for i in DateCols];
_ = _.astype(np.int8);
Xtrain = pd.concat([Xtrain.drop(['Holiday_Nm'], axis=1, errors= 'ignore'), 
                    pd.DataFrame(_, index= Xtrain.index, columns = DateCols)], axis=1);

_ = OHencoder.transform(Xtest);
_ = _.astype(np.int8);
Xtest = pd.concat([Xtest.drop(['Holiday_Nm'], axis=1, errors= 'ignore'), 
                   pd.DataFrame(_, index= Xtest.index, columns = DateCols)], axis=1);

# Preparing controls:-
PrintColor(f"\nData pipeline output columns\n");
display(np.array(Xtrain.columns));

PrintColor(f"\nData pipeline checks\n");
PrintColor(f"Input lengths={len(xtrain)} {len(xtest)} and Output lengths={len(Xtrain)} {len(Xtest)}",
           attrs= ['dark']);

del xtrain, _;

for i in range(3): collect(i);

## C. Correlation plot

We assess the feature correlations after feature creation to better study their interactions

In [ ]:
def MakeCorrPlot(corr_req:str= "N"):
    "This function makes the train set correlation plot if requested by the user";
    
    if corr_req.upper() == "Y":
        _ = Xtrain.corr();

        fig, ax = plt.subplots(1,1, figsize= (40, 30));
        heatmap(_, annot= True, fmt= '.0%', linewidth= 0.35, center= True, cmap= 'icefire', cbar= False,
                linecolor= 'white', mask = np.triu(np.ones_like(_)),ax= ax);
        ax.set_title(f"\nCorrelation plots after feature creation\n", **title_specs);

        plt.tight_layout();
        plt.show();

        del _; 
        collect();

In [ ]:
MakeCorrPlot(corr_req= "Y");

In [ ]:
# Saving relevant datasets:-
Xtrain.to_csv('Xtrain.csv');
Xtest.to_csv('Xtest.csv');

## D. Developing the training master data-table

We now join the date features and targets and develop the train-set for model development

In [ ]:
Train_Mst = pd.concat([Xtrain, Sales_Prf.iloc[:, -48:]], axis=1);

# Saving relevant datasets:-
Xtrain.to_csv('Xtrain.csv');
Xtest.to_csv('Xtest.csv');
Train_Mst.to_csv("Train_Mst.csv");

We approach the end of the data engineering section. We now develop models using these features and calibrate them appropriately.